In [1]:
import pandas as pd 
import numpy as np 
import plotly.graph_objects as go 
import plotly.express as px 
from scipy.stats import stats

In [2]:
pd.set_option("display.max_columns",None)

In [3]:
df = pd.read_csv(r"C:\works\learnings\Decision_tree\banking_market\bank-additional-full.csv", sep=";")
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,261,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,149,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,226,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,151,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,307,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [4]:
df.describe()

,age,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
count,41188.00000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000
mean,40.02406,258.285010,2.567593,962.475454,0.172963,0.081886,93.575664,-40.502600,3.621291,5167.035911
std,10.42125,259.279249,2.770014,186.910907,0.494901,1.570960,0.578840,4.628198,1.734447,72.251528
min,17.00000,0.000000,1.000000,0.000000,0.000000,-3.400000,92.201000,-50.800000,0.634000,4963.600000
25%,32.00000,102.000000,1.000000,999.000000,0.000000,-1.800000,93.075000,-42.700000,1.344000,5099.100000
50%,38.00000,180.000000,2.000000,999.000000,0.000000,1.100000,93.749000,-41.800000,4.857000,5191.000000
75%,47.00000,319.000000,3.000000,999.000000,0.000000,1.400000,93.994000,-36.400000,4.961000,5228.100000
max,98.00000,4918.000000,56.000000,999.000000,7.000000,1.400000,94.767000,-26.900000,5.045000,5228.100000


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   marital         41188 non-null  object 
 3   education       41188 non-null  object 
 4   default         41188 non-null  object 
 5   housing         41188 non-null  object 
 6   loan            41188 non-null  object 
 7   contact         41188 non-null  object 
 8   month           41188 non-null  object 
 9   day_of_week     41188 non-null  object 
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  object 
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null 

In [6]:
cat_columns = df.select_dtypes(include=["object"]).columns
cat_columns

Index(['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact',
       'month', 'day_of_week', 'poutcome', 'y'],
      dtype='object')

In [7]:

for col in cat_columns:
    print(df[col].value_counts())

job
admin.           10422
blue-collar       9254
technician        6743
services          3969
management        2924
retired           1720
entrepreneur      1456
self-employed     1421
housemaid         1060
unemployed        1014
student            875
unknown            330
Name: count, dtype: int64
marital
married     24928
single      11568
divorced     4612
unknown        80
Name: count, dtype: int64
education
university.degree      12168
high.school             9515
basic.9y                6045
professional.course     5243
basic.4y                4176
basic.6y                2292
unknown                 1731
illiterate                18
Name: count, dtype: int64
default
no         32588
unknown     8597
yes            3
Name: count, dtype: int64
housing
yes        21576
no         18622
unknown      990
Name: count, dtype: int64
loan
no         33950
yes         6248
unknown      990
Name: count, dtype: int64
contact
cellular     26144
telephone    15044
Name: count, dtype: in

In [8]:
not_selected = ['contact','month', 'day_of_week', "y"]
cat_columns = [ col for col in cat_columns if col not in not_selected]
cat_columns

['job', 'marital', 'education', 'default', 'housing', 'loan', 'poutcome']

In [9]:
num_columns = df.select_dtypes(include=["int64","float64"]).columns
num_columns

Index(['age', 'duration', 'campaign', 'pdays', 'previous', 'emp.var.rate',
       'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed'],
      dtype='object')

In [10]:
df.shape

(41188, 21)

In [11]:
from scipy.stats import chi2_contingency

dec= {}
for col in cat_columns:
    ct = pd.crosstab(df[col],df["y"])
    chi_2 , p , dof, exp = chi2_contingency(ct)
    dec[col] = p
    print(f"{col:15} p-value : {p:.5f}")

job             p-value : 0.00000
marital         p-value : 0.00000
education       p-value : 0.00000
default         p-value : 0.00000
housing         p-value : 0.05829
loan            p-value : 0.57868
poutcome        p-value : 0.00000


In [12]:
cat_columns = ['job', 'marital', 'education', 'housing', 'loan', 'poutcome']

In [13]:
for key,pvalue in dec.items():
    if pvalue < 0.05 :
        print(f"{key} : We have enough evidence to reject the H0")
    else :
        print(f"{key} : We dont have enough evidence to reject the H0")

job : We have enough evidence to reject the H0
marital : We have enough evidence to reject the H0
education : We have enough evidence to reject the H0
default : We have enough evidence to reject the H0
housing : We dont have enough evidence to reject the H0
loan : We dont have enough evidence to reject the H0
poutcome : We have enough evidence to reject the H0


**default = yes → 3 samples**
This violates Chi-Square assumptions:

Expected cell counts should be ≥ 5

Extreme imbalance → inflated χ²

**Expect for default we are selection since from our observation in analysis we got to know that defauolt have y = 3 which is highe imbalanced for that reason we are dropping it**

In [14]:
num_dec = {}
for col in num_columns:
    num_dec[col] = {}
    num_dec[col]["skewness"] = df[col].skew()
    num_dec[col]["kurtosis"] = df[col].kurtosis()
num_dec

{'age': {'skewness': 0.7846968157646645, 'kurtosis': 0.7913115311544336},
 'duration': {'skewness': 3.263141255262832, 'kurtosis': 20.247938014978796},
 'campaign': {'skewness': 4.762506697067009, 'kurtosis': 36.979795142898865},
 'pdays': {'skewness': -4.922189916418162, 'kurtosis': 22.22946262635535},
 'previous': {'skewness': 3.8320422428611836, 'kurtosis': 20.108816215208236},
 'emp.var.rate': {'skewness': -0.7240955492472556,
  'kurtosis': -1.0626315246508407},
 'cons.price.idx': {'skewness': -0.23088765135788006,
  'kurtosis': -0.8298085771833406},
 'cons.conf.idx': {'skewness': 0.30317985874819237,
  'kurtosis': -0.35855831054052567},
 'euribor3m': {'skewness': -0.7091879563778298,
  'kurtosis': -1.4068026223874996},
 'nr.employed': {'skewness': -1.044262407089151,
  'kurtosis': -0.0037603756956321455}}

**For both skewness and kurtosis rnage from 0~1 is the optimal range if the value greater then the ranage then we have heavy tail and better to avoid those feature**

In [15]:
from scipy.stats import mannwhitneyu

mw_results = {}

for col in num_columns:
    group_no = df[df['y'] == 'no'][col]
    group_yes = df[df['y'] == 'yes'][col]

    u_stat, p_value = mannwhitneyu(group_no, group_yes, alternative='two-sided')
    mw_results[col] = p_value

    print(f"{col:15} p-value = {p_value:.5f}")

age             p-value = 0.01608
duration        p-value = 0.00000
campaign        p-value = 0.00000
pdays           p-value = 0.00000
previous        p-value = 0.00000
emp.var.rate    p-value = 0.00000
cons.price.idx  p-value = 0.00000
cons.conf.idx   p-value = 0.00000
euribor3m       p-value = 0.00000
nr.employed     p-value = 0.00000


In [16]:
df["duration"]

0        261
1        149
2        226
3        151
4        307
        ... 
41183    334
41184    383
41185    189
41186    442
41187    239
Name: duration, Length: 41188, dtype: int64

**We are dropping the duration feature since its the call duration which lead to data leage in the model i.e after contacting we will get to know about the duration if we gove the duration to model then we are giving the target likely for that reason we cant use the duration feature**

In [24]:
num_columns = ['age', 'campaign',  'previous', 'emp.var.rate',
       'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']

In [25]:
new_df = pd.concat([df[cat_columns],df[num_columns]], axis=1)
print(new_df.shape)
new_df.columns

(41188, 14)


Index(['job', 'marital', 'education', 'housing', 'loan', 'poutcome', 'age',
       'campaign', 'previous', 'emp.var.rate', 'cons.price.idx',
       'cons.conf.idx', 'euribor3m', 'nr.employed'],
      dtype='object')

In [26]:
final_df = pd.get_dummies(new_df, columns=cat_columns, drop_first=True)
final_df.columns

Index(['age', 'campaign', 'previous', 'emp.var.rate', 'cons.price.idx',
       'cons.conf.idx', 'euribor3m', 'nr.employed', 'job_blue-collar',
       'job_entrepreneur', 'job_housemaid', 'job_management', 'job_retired',
       'job_self-employed', 'job_services', 'job_student', 'job_technician',
       'job_unemployed', 'job_unknown', 'marital_married', 'marital_single',
       'marital_unknown', 'education_basic.6y', 'education_basic.9y',
       'education_high.school', 'education_illiterate',
       'education_professional.course', 'education_university.degree',
       'education_unknown', 'housing_unknown', 'housing_yes', 'loan_unknown',
       'loan_yes', 'poutcome_nonexistent', 'poutcome_success'],
      dtype='object')

In [27]:
final_df.head()

,age,campaign,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,job_blue-collar,job_entrepreneur,job_housemaid,job_management,job_retired,job_self-employed,job_services,job_student,job_technician,job_unemployed,job_unknown,marital_married,marital_single,marital_unknown,education_basic.6y,education_basic.9y,education_high.school,education_illiterate,education_professional.course,education_university.degree,education_unknown,housing_unknown,housing_yes,loan_unknown,loan_yes,poutcome_nonexistent,poutcome_success
0,56,1,0,1.1,93.994,-36.4,4.857,5191.0,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
1,57,1,0,1.1,93.994,-36.4,4.857,5191.0,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False
2,37,1,0,1.1,93.994,-36.4,4.857,5191.0,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,True,False,False,True,False
3,40,1,0,1.1,93.994,-36.4,4.857,5191.0,False,False,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False
4,56,1,0,1.1,93.994,-36.4,4.857,5191.0,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,True,True,False


In [28]:
final_df.shape

(41188, 35)

In [32]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import KFold
clf = DecisionTreeClassifier(
    max_depth=5,
    random_state= 42,
    criterion="gini",
    min_samples_split=20
)

kf = KFold(n_splits=5, shuffle= True, random_state=42)
x = final_df
y = df["y"]

result = []
for train_idx, val_idx in kf.split(x):
    x_train, x_val = x.iloc[train_idx], x.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    clf.fit(x_train,y_train)
    score = clf.score(x_val,y_val)
    result.append(score)

result

[0.896698227725176,
 0.8997329448895363,
 0.8931779558145181,
 0.9027558577151875,
 0.8951074420298653]